In [124]:
import pandas as pd
import pymysql
import urllib

In [125]:
def find_closest_mapping_session(release, mapping_df, direction='above'):
    """
    Find the mapping_session_id with the closest old_release to a given release.
    
    Parameters:
    -----------
    release : float or int
        The release number to find the closest match for
    mapping_df : pandas.DataFrame
        The mapping_session dataframe (default: map_df_115)
    direction : str
        'below' - find the closest release below the given release
        'above' - find the closest release above the given release
    
    Returns:
    --------
    pandas.DataFrame with the matching row(s)
    """
    mapping_df = mapping_df.copy()
    mapping_df['abs_diff'] = abs(mapping_df['old_release'] - release)
    
    if direction == 'below':
        # Find closest release below (old_release < release)
        below_releases = mapping_df[mapping_df['old_release'] <= release]
        if len(below_releases) > 0:
            min_diff = below_releases['abs_diff'].min()
            closest_releases = below_releases[below_releases['abs_diff'] == min_diff]
            latest_session = closest_releases['created'].max()
            result = closest_releases[closest_releases['created'] == latest_session]
        else:
            result = pd.DataFrame()  # No releases below
            print(f"No releases below {release} found")
    elif direction == 'above':
        # Find closest release above (old_release > release)
        above_releases = mapping_df[mapping_df['old_release'] >= release]
        if len(above_releases) > 0:
            min_diff = above_releases['abs_diff'].min()
            closest_releases = above_releases[above_releases['abs_diff'] == min_diff]
            earliest_session = closest_releases['created'].min()
            result = closest_releases[closest_releases['created'] == earliest_session]
        else:
            result = pd.DataFrame()  # No releases above
            print(f"No releases above {release} found")
    else:
        raise ValueError("direction must be 'below', or 'above'")
    
    # Clean up and return relevant columns
    if len(result) > 0:
        result = result[['mapping_session_id', 'old_db_name', 'new_db_name', 
                        'old_release', 'new_release', 'old_assembly', 'new_assembly', 'created']]
        result = result.drop_duplicates()
    
    return result

In [126]:
# Simpler helper function that returns just the mapping_session_id
def get_mapping_session_id(release, mapping_df, direction='below'):
    """
    Get the mapping_session_id for the closest old_release.
    
    Parameters:
    -----------
    release : float or int
        The release number
    mapping_df : pandas.DataFrame
        The mapping_session dataframe (default: map_df_115)
    direction : str
        'below', or 'above'
    
    Returns:
    --------
    int or list of ints: mapping_session_id(s)
    """
    result = find_closest_mapping_session(release, mapping_df, direction)
    if len(result) == 0:
        return None
    elif len(result) == 1:
        return result['mapping_session_id'].iloc[0]
    else:
        return result['mapping_session_id'].tolist()

In [127]:
initial_release = 91
latest_release = 115

In [128]:
# Read the annotation file with pandas
annotation_file = "./GSM4150378_sciPlex3_A549_MCF7_K562_screen_gene.annotations.txt"

annotations_df = pd.read_csv(annotation_file, sep=' ', low_memory=False)

In [129]:
annotations_df['stable_id'] = annotations_df['id'].str.split('.').str[0]
annotations_df['version'] = annotations_df['id'].str.split('.').str[1]

In [130]:
annotations_df = annotations_df[annotations_df['id'].str.startswith('ENSG')]

In [131]:
base_url = f"https://ftp.ensembl.org/pub/release-{latest_release}/mysql/homo_sapiens_core_{latest_release}_38/"


In [132]:
base_url

'https://ftp.ensembl.org/pub/release-115/mysql/homo_sapiens_core_115_38/'

In [133]:
map_table_file = "mapping_session.txt.gz"
map_table_url = base_url + map_table_file


print(f"Downloading gene table: {map_table_url}...")
urllib.request.urlretrieve(map_table_url, map_table_file)
print(f"Downloaded: {map_table_file}")

map_df = pd.read_csv(map_table_file, sep='\t', compression='gzip', 
                      header=None, low_memory=False,
                      na_values=['\\N', 'NULL'], keep_default_na=True)

map_columns = ['mapping_session_id', 'old_db_name', 'new_db_name', 'old_release', 'new_release', 'old_assembly', 'new_assembly', 'created', ]
map_df.columns = map_columns

Downloaded: mapping_session.txt.gz


In [134]:
event_table_file = "stable_id_event.txt.gz"
event_table_url = base_url + event_table_file

print(f"Downloading gene table: {event_table_url}...")
urllib.request.urlretrieve(event_table_url, event_table_file)
print(f"Downloaded: {event_table_file}")

event_df = pd.read_csv(event_table_file, sep='\t', compression='gzip', 
                      header=None, low_memory=False,
                      na_values=['\\N', 'NULL'], keep_default_na=True)

event_columns = ['old_stable_id', 'old_version', 'new_stable_id', 'new_version', 'mapping_session_id', 'type', 'score' ]
event_df.columns = event_columns

Downloaded: stable_id_event.txt.gz


In [135]:
map_df['created'].min()

'2002-09-03 11:19:48'

In [136]:
# Example: Get mapping_session_id for release 91
source_release = 91
source_session_id = get_mapping_session_id(source_release, map_df)

In [137]:
# Example: Get mapping_session_id for release 91
target_release = 115
target_session_id = get_mapping_session_id(target_release, map_df, direction='below')

In [138]:
target_session_id

np.int64(425)

In [139]:
source_date = map_df[map_df['mapping_session_id']==source_session_id]['created'].item()

In [140]:
target_date = map_df[map_df['mapping_session_id']==target_session_id]['created'].item()

In [141]:
map_df_slice = map_df[(map_df['created']>=source_date)&(map_df['created']<=target_date)]

In [142]:
assert map_df_slice.sort_values(by=['created']).equals(map_df_slice.sort_values(by=['old_release']))

In [143]:
map_df_slice = map_df_slice.sort_values(by=['created'])

In [144]:
map_df_slice['mapping_session_id'].values

array([404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416,
       417, 418, 419, 420, 421, 422, 423, 424, 425])

In [145]:
old_stable_ids = event_df[event_df['mapping_session_id'] == 404]['old_stable_id'].values
new_stable_ids = event_df[event_df['mapping_session_id'] == 404]['new_stable_id'].values

In [147]:
# Download gene table from initial release (91)
initial_gene_table_url = f"https://ftp.ensembl.org/pub/release-{initial_release}/mysql/homo_sapiens_core_{initial_release}_38/gene.txt.gz"
initial_gene_table_file = f"gene_{initial_release}.txt.gz"

print(f"Downloading gene table from release {initial_release}: {initial_gene_table_url}...")
urllib.request.urlretrieve(initial_gene_table_url, initial_gene_table_file)
print(f"Downloaded: {initial_gene_table_file}")

# Load gene table with proper columns
gene_columns = ['gene_id', 'biotype', 'analysis_id', 'seq_region_id', 
                'seq_region_start', 'seq_region_end', 'seq_region_strand',
                'display_xref_id', 'source', 'description', 'is_current',
                'canonical_transcript_id', 'stable_id', 'version', 'created_date', 'modified_date']

initial_gene_df = pd.read_csv(initial_gene_table_file, sep='\t', compression='gzip', 
                              header=None, low_memory=False,
                              na_values=['\\N', 'NULL'], keep_default_na=True)

initial_gene_df.columns = gene_columns

print(f"\nLoaded {len(initial_gene_df)} genes from release {initial_release}")
print(f"Columns: {list(initial_gene_df.columns)}")
print(f"\nFirst few rows:")
print(initial_gene_df[['gene_id', 'stable_id', 'version']].head())


Downloaded: gene_91.txt.gz

Loaded 64661 genes from release 91
Columns: ['gene_id', 'biotype', 'analysis_id', 'seq_region_id', 'seq_region_start', 'seq_region_end', 'seq_region_strand', 'display_xref_id', 'source', 'description', 'is_current', 'canonical_transcript_id', 'stable_id', 'version', 'created_date', 'modified_date']

First few rows:
   gene_id        stable_id  version
0   189354  ENSG00000283891        1
1   189353  ENSG00000251931        1
2   189351  ENSG00000207766        1
3   189352  ENSG00000275323        3
4   189350  ENSG00000276678        1


In [148]:
# Initialize the mapping: start with genes from initial release
# Create a mapping dictionary: {release_91_stable_id: current_stable_id}
current_mapping = dict(zip(initial_gene_df['stable_id'], initial_gene_df['stable_id']))

print(f"Initial mapping: {len(current_mapping)} genes from release {initial_release}")
print(f"Sample mapping (first 5):")
for i, (k, v) in enumerate(list(current_mapping.items())[:5]):
    print(f"  {k} -> {v}")

# Get the list of mapping sessions to iterate through
mapping_sessions = map_df_slice['mapping_session_id'].values
print(f"\nWill iterate through {len(mapping_sessions)} mapping sessions:")
print(mapping_sessions)


Initial mapping: 64661 genes from release 91
Sample mapping (first 5):
  ENSG00000283891 -> ENSG00000283891
  ENSG00000251931 -> ENSG00000251931
  ENSG00000207766 -> ENSG00000207766
  ENSG00000275323 -> ENSG00000275323
  ENSG00000276678 -> ENSG00000276678

Will iterate through 22 mapping sessions:
[404 405 406 407 408 409 410 411 412 413 414 415 416 417 418 419 420 421
 422 423 424 425]


In [149]:
# Iterate through each mapping session to update the mapping
mapping_history = []  # Track changes at each step

for session_id in mapping_sessions:
    print(f"\n{'='*80}")
    print(f"Processing mapping_session_id: {session_id}")
    
    # Get mapping session info
    session_info = map_df_slice[map_df_slice['mapping_session_id'] == session_id].iloc[0]
    print(f"  Mapping from release {session_info['old_release']} to {session_info['new_release']}")
    print(f"  Old DB: {session_info['old_db_name']} -> New DB: {session_info['new_db_name']}")
    
    # Get stable_id_event entries for this mapping session
    session_events = event_df[event_df['mapping_session_id'] == session_id].copy()
    print(f"  Found {len(session_events)} stable_id events")
    
    # Create a mapping dictionary for this session: old_stable_id -> new_stable_id
    # Also track deprecation: old_stable_id -> None/NaN (deprecated)
    session_mapping = {}
    deprecated_ids = set()  # Track IDs that are deprecated in this session
    
    for _, row in session_events.iterrows():
        old_id = row['old_stable_id']
        new_id = row['new_stable_id']
        
        if pd.notna(old_id):
            if pd.notna(new_id):
                # Valid mapping: old_id -> new_id
                session_mapping[old_id] = new_id
            else:
                # Deprecation: old_id exists but new_id is None/NaN -> gene is deprecated
                deprecated_ids.add(old_id)
    
    print(f"  Created mapping for {len(session_mapping)} stable IDs")
    print(f"  Found {len(deprecated_ids)} deprecated IDs")
    
    # Update current_mapping: for each gene, handle both updates and deprecation
    updated_count = 0
    deprecated_count = 0
    
    for original_id, current_id in current_mapping.items():
        # Skip if already deprecated (stay deprecated)
        if pd.isna(current_id) or current_id is None:
            continue
            
        # Check if this gene is deprecated in this session
        if current_id in deprecated_ids:
            current_mapping[original_id] = None  # Mark as deprecated
            deprecated_count += 1
        # Check if this gene maps to a new ID
        elif current_id in session_mapping:
            new_id = session_mapping[current_id]
            current_mapping[original_id] = new_id
            updated_count += 1
    
    print(f"  Updated {updated_count} gene mappings")
    print(f"  Deprecated {deprecated_count} genes")
    
    # Track history
    mapping_history.append({
        'session_id': session_id,
        'old_release': session_info['old_release'],
        'new_release': session_info['new_release'],
        'events': len(session_events),
        'updated': updated_count,
        'deprecated': deprecated_count
    })

print(f"\n{'='*80}")
print("Mapping complete!")
print(f"Final mapping: {len(current_mapping)} genes mapped from release {initial_release} to {latest_release}")



Processing mapping_session_id: 404
  Mapping from release 91.0 to 92.0
  Old DB: homo_sapiens_core_91_38 -> New DB: homo_sapiens_core_92_38
  Found 32387 stable_id events
  Created mapping for 22229 stable IDs
  Found 659 deprecated IDs
  Updated 7283 gene mappings
  Deprecated 174 genes

Processing mapping_session_id: 405
  Mapping from release 93.0 to 94.0
  Old DB: homo_sapiens_core_93_38 -> New DB: homo_sapiens_core_94_38
  Found 9575 stable_id events
  Created mapping for 4180 stable IDs
  Found 400 deprecated IDs
  Updated 1879 gene mappings
  Deprecated 120 genes

Processing mapping_session_id: 406
  Mapping from release 95.0 to 96.0
  Old DB: homo_sapiens_core_95_38 -> New DB: homo_sapiens_core_96_38
  Found 38012 stable_id events
  Created mapping for 34078 stable IDs
  Found 514 deprecated IDs
  Updated 11292 gene mappings
  Deprecated 121 genes

Processing mapping_session_id: 407
  Mapping from release 96.0 to 97.0
  Old DB: homo_sapiens_core_96_38 -> New DB: homo_sapiens_c

In [106]:
# Create final mapping DataFrame
final_mapping_df = pd.DataFrame({
    'release_91_stable_id': list(current_mapping.keys()),
    'release_115_stable_id': list(current_mapping.values())
})

print(f"Final mapping DataFrame:")
print(f"  Shape: {final_mapping_df.shape}")
print(f"  Columns: {list(final_mapping_df.columns)}")
print(f"\nFirst 10 rows:")
print(final_mapping_df.head(10))

# Check for unmapped genes (genes that don't exist in release 115)
# These might be genes that were deleted or merged
unmapped = final_mapping_df[final_mapping_df['release_115_stable_id'].isna()]
print(f"\nUnmapped genes (None values): {len(unmapped)}")

# Check for genes that mapped to the same ID (no change)
unchanged = final_mapping_df[final_mapping_df['release_91_stable_id'] == final_mapping_df['release_115_stable_id']]
print(f"Unchanged genes (same ID in both releases): {len(unchanged)}")

# Check for genes that changed
changed = final_mapping_df[final_mapping_df['release_91_stable_id'] != final_mapping_df['release_115_stable_id']]
print(f"Changed genes (different ID in release 115): {len(changed)}")

# Show mapping history summary
print(f"\n{'='*80}")
print("Mapping History Summary:")
print(f"{'='*80}")
history_df = pd.DataFrame(mapping_history)
print(history_df)


Final mapping DataFrame:
  Shape: (64661, 2)
  Columns: ['release_91_stable_id', 'release_115_stable_id']

First 10 rows:
  release_91_stable_id release_115_stable_id
0      ENSG00000283891       ENSG00000283891
1      ENSG00000251931       ENSG00000251931
2      ENSG00000207766       ENSG00000207766
3      ENSG00000275323       ENSG00000275323
4      ENSG00000276678       ENSG00000276678
5      ENSG00000207260       ENSG00000207260
6      ENSG00000265993       ENSG00000265993
7      ENSG00000207185       ENSG00000207185
8      ENSG00000283793       ENSG00000283793
9      ENSG00000201545       ENSG00000201545

Unmapped genes (None values): 994
Unchanged genes (same ID in both releases): 62854
Changed genes (different ID in release 115): 1807

Mapping History Summary:
    session_id  old_release  new_release  events  updated  deprecated
0          404         91.0         92.0   32387     7283         174
1          405         93.0         94.0    9575     1879         120
2          4

In [107]:
# Add version information from initial release

final_mapping_df = final_mapping_df.merge(
    initial_gene_df[['stable_id', 'version']], 
    left_on='release_91_stable_id', 
    right_on='stable_id', 
    how='left'
)
final_mapping_df = final_mapping_df.rename(columns={'version': 'release_91_version'})
final_mapping_df = final_mapping_df.drop(columns=['stable_id'])

# Create full Ensembl IDs with versions
final_mapping_df['release_91_full_id'] = (
    final_mapping_df['release_91_stable_id'].astype(str) + '.' + 
    final_mapping_df['release_91_version'].astype(str)
)

# Download gene table from latest release to get versions
latest_gene_table_url = f"https://ftp.ensembl.org/pub/release-{latest_release}/mysql/homo_sapiens_core_{latest_release}_38/gene.txt.gz"
latest_gene_table_file = f"gene_{latest_release}.txt.gz"

print(f"Downloading gene table from release {latest_release}: {latest_gene_table_url}...")
urllib.request.urlretrieve(latest_gene_table_url, latest_gene_table_file)
print(f"Downloaded: {latest_gene_table_file}")

latest_gene_df = pd.read_csv(latest_gene_table_file, sep='\t', compression='gzip', 
                             header=None, low_memory=False,
                             na_values=['\\N', 'NULL'], keep_default_na=True)
latest_gene_df.columns = gene_columns

# Add version information from latest release
final_mapping_df = final_mapping_df.merge(
    latest_gene_df[['stable_id', 'version']], 
    left_on='release_115_stable_id', 
    right_on='stable_id', 
    how='left'
)
final_mapping_df = final_mapping_df.rename(columns={'version': 'release_115_version'})
final_mapping_df = final_mapping_df.drop(columns=['stable_id'])

# Create full Ensembl IDs with versions for release 115
final_mapping_df['release_115_full_id'] = (
    final_mapping_df['release_115_stable_id'].astype(str) + '.' + 
    final_mapping_df['release_115_version'].astype(str)
)

# Reorder columns
final_mapping_df = final_mapping_df[[
    'release_91_stable_id', 'release_91_version', 'release_91_full_id',
    'release_115_stable_id', 'release_115_version', 'release_115_full_id'
]]

print(f"\nFinal mapping with versions:")
print(final_mapping_df.head(10))
print(f"\nTotal mappings: {len(final_mapping_df)}")
print(f"Successfully mapped: {len(final_mapping_df[final_mapping_df['release_115_stable_id'].notna()])}")
print(f"Unmapped: {len(final_mapping_df[final_mapping_df['release_115_stable_id'].isna()])}")


Downloaded: gene_115.txt.gz

Final mapping with versions:
  release_91_stable_id  release_91_version release_91_full_id  \
0      ENSG00000283891                   1  ENSG00000283891.1   
1      ENSG00000251931                   1  ENSG00000251931.1   
2      ENSG00000207766                   1  ENSG00000207766.1   
3      ENSG00000275323                   3  ENSG00000275323.3   
4      ENSG00000276678                   1  ENSG00000276678.1   
5      ENSG00000207260                   1  ENSG00000207260.1   
6      ENSG00000265993                   1  ENSG00000265993.1   
7      ENSG00000207185                   1  ENSG00000207185.1   
8      ENSG00000283793                   1  ENSG00000283793.1   
9      ENSG00000201545                   1  ENSG00000201545.1   

  release_115_stable_id  release_115_version  release_115_full_id  
0       ENSG00000283891                  1.0  ENSG00000283891.1.0  
1       ENSG00000251931                  1.0  ENSG00000251931.1.0  
2       ENSG0000020776

In [108]:
# Save the final mapping to a file
output_file = f"ensembl_mapping_{initial_release}_to_{latest_release}.tsv"
final_mapping_df.to_csv(output_file, sep='\t', index=False)
print(f"Saved final mapping to: {output_file}")

# Also save a summary
summary_file = f"ensembl_mapping_{initial_release}_to_{latest_release}_summary.txt"
with open(summary_file, 'w') as f:
    f.write(f"Ensembl ID Mapping: Release {initial_release} to Release {latest_release}\n")
    f.write("="*80 + "\n\n")
    f.write(f"Total genes in release {initial_release}: {len(initial_gene_df)}\n")
    f.write(f"Total genes successfully mapped: {len(final_mapping_df[final_mapping_df['release_115_stable_id'].notna()])}\n")
    f.write(f"Unmapped genes: {len(final_mapping_df[final_mapping_df['release_115_stable_id'].isna()])}\n")
    f.write(f"Unchanged genes (same ID): {len(final_mapping_df[final_mapping_df['release_91_stable_id'] == final_mapping_df['release_115_stable_id']])}\n")
    f.write(f"Changed genes (different ID): {len(final_mapping_df[final_mapping_df['release_91_stable_id'] != final_mapping_df['release_115_stable_id']])}\n")
    f.write(f"\nMapping sessions processed: {len(mapping_sessions)}\n")
    f.write("\nMapping History:\n")
    f.write(history_df.to_string(index=False))

print(f"Saved summary to: {summary_file}")

print(f"\n{'='*80}")
print("Mapping complete! Files saved:")
print(f"  - {output_file}")
print(f"  - {summary_file}")
print(f"{'='*80}")


Saved final mapping to: ensembl_mapping_91_to_115.tsv
Saved summary to: ensembl_mapping_91_to_115_summary.txt

Mapping complete! Files saved:
  - ensembl_mapping_91_to_115.tsv
  - ensembl_mapping_91_to_115_summary.txt


In [109]:
final_mapping_df

,release_91_stable_id,release_91_version,release_91_full_id,release_115_stable_id,release_115_version,release_115_full_id
0,ENSG00000283891,1,ENSG00000283891.1,ENSG00000283891,1.0,ENSG00000283891.1.0
1,ENSG00000251931,1,ENSG00000251931.1,ENSG00000251931,1.0,ENSG00000251931.1.0
2,ENSG00000207766,1,ENSG00000207766.1,ENSG00000207766,1.0,ENSG00000207766.1.0
3,ENSG00000275323,3,ENSG00000275323.3,ENSG00000275323,3.0,ENSG00000275323.3.0
4,ENSG00000276678,1,ENSG00000276678.1,ENSG00000276678,1.0,ENSG00000276678.1.0
...,...,...,...,...,...,...
64656,LRG_995,1,LRG_995.1,LRG_995,1.0,LRG_995.1.0
64657,LRG_996,1,LRG_996.1,LRG_996,1.0,LRG_996.1.0
64658,LRG_997,1,LRG_997.1,LRG_997,1.0,LRG_997.1.0
64659,LRG_998,1,LRG_998.1,LRG_998,1.0,LRG_998.1.0


In [110]:
initial_gene_df['stable_id'].isin(final_mapping_df['release_91_stable_id'].unique()).sum()

np.int64(64661)

In [111]:
final_mapping_df[~final_mapping_df['release_115_stable_id'].isin(latest_gene_df['stable_id'])]

,release_91_stable_id,release_91_version,release_91_full_id,release_115_stable_id,release_115_version,release_115_full_id
231,ENSG00000279049,1,ENSG00000279049.1,None,NaN,None.nan
266,ENSG00000281557,1,ENSG00000281557.1,None,NaN,None.nan
442,ENSG00000268991,2,ENSG00000268991.2,None,NaN,None.nan
495,ENSG00000247732,2,ENSG00000247732.2,None,NaN,None.nan
560,ENSG00000241978,9,ENSG00000241978.9,None,NaN,None.nan
...,...,...,...,...,...,...
63925,ENSG00000284381,1,ENSG00000284381.1,None,NaN,None.nan
63928,ENSG00000284004,1,ENSG00000284004.1,None,NaN,None.nan
63949,ENSG00000284208,1,ENSG00000284208.1,None,NaN,None.nan
63962,ENSG00000284354,1,ENSG00000284354.1,None,NaN,None.nan


In [123]:
final_mapping_df[final_mapping_df['release_115_stable_id'].isna()]['release_91_stable_id'].isin(latest_gene_df['stable_id'])

231      False
266      False
442      False
495      False
560      False
         ...  
63925    False
63928    False
63949    False
63962    False
63965    False
Name: release_91_stable_id, Length: 994, dtype: bool

In [70]:
latest_gene_df[latest_gene_df['stable_id'] == 'ENSG00000281557']

,gene_id,biotype,analysis_id,seq_region_id,seq_region_start,seq_region_end,seq_region_strand,display_xref_id,source,description,is_current,canonical_transcript_id,stable_id,version,created_date,modified_date


In [63]:
64661 - 977

63684

In [55]:
#latest_gene_df[latest_gene_df['stable_id'] == 'ENSG00000284354']

In [ ]:
# Download gene table from initial release (91)
initial_gene_table_url = f"https://ftp.ensembl.org/pub/release-{initial_release}/mysql/homo_sapiens_core_{initial_release}_38/gene.txt.gz"
initial_gene_table_file = f"gene_{initial_release}.txt.gz"

print(f"Downloading gene table from release {initial_release}: {initial_gene_table_url}...")
urllib.request.urlretrieve(initial_gene_table_url, initial_gene_table_file)
print(f"Downloaded: {initial_gene_table_file}")

# Load gene table with proper columns
gene_columns = ['gene_id', 'biotype', 'analysis_id', 'seq_region_id', 
                'seq_region_start', 'seq_region_end', 'seq_region_strand',
                'display_xref_id', 'source', 'description', 'is_current',
                'canonical_transcript_id', 'stable_id', 'version', 'created_date', 'modified_date']

initial_gene_df = pd.read_csv(initial_gene_table_file, sep='\t', compression='gzip', 
                              header=None, low_memory=False,
                              na_values=['\\N', 'NULL'], keep_default_na=True)

initial_gene_df.columns = gene_columns

print(f"\nLoaded {len(initial_gene_df)} genes from release {initial_release}")
print(f"Columns: {list(initial_gene_df.columns)}")
print(f"\nFirst few rows:")
print(initial_gene_df[['gene_id', 'stable_id', 'version']].head())


In [73]:
# Download gene table from initial release (91)
gene_table_url = f"https://ftp.ensembl.org/pub/release-{114}/mysql/homo_sapiens_core_{114}_38/gene.txt.gz"
gene_table_file = f"gene_{114}.txt.gz"

print(f"Downloading gene table from release {114}: {gene_table_url}...")
urllib.request.urlretrieve(gene_table_url, gene_table_file)
print(f"Downloaded: {gene_table_file}")

# Load gene table with proper columns
gene_columns = ['gene_id', 'biotype', 'analysis_id', 'seq_region_id', 
                'seq_region_start', 'seq_region_end', 'seq_region_strand',
                'display_xref_id', 'source', 'description', 'is_current',
                'canonical_transcript_id', 'stable_id', 'version', 'created_date', 'modified_date']

gene_df = pd.read_csv(gene_table_file, sep='\t', compression='gzip', 
                              header=None, low_memory=False,
                              na_values=['\\N', 'NULL'], keep_default_na=True)

gene_df.columns = gene_columns

print(f"\nLoaded {len(gene_df)} genes from release {114}")
print(f"Columns: {list(gene_df.columns)}")
print(f"\nFirst few rows:")
print(gene_df[['gene_id', 'stable_id', 'version']].head())

Downloaded: gene_114.txt.gz

Loaded 87688 genes from release 114
Columns: ['gene_id', 'biotype', 'analysis_id', 'seq_region_id', 'seq_region_start', 'seq_region_end', 'seq_region_strand', 'display_xref_id', 'source', 'description', 'is_current', 'canonical_transcript_id', 'stable_id', 'version', 'created_date', 'modified_date']

First few rows:
   gene_id        stable_id  version
0      554  ENSG00000210049        1
1      555  ENSG00000211459        2
2      556  ENSG00000210077        1
3      557  ENSG00000210082        2
4      558  ENSG00000209082        1


In [74]:
gene_df[gene_df['stable_id'] == 'ENSG00000281557']

,gene_id,biotype,analysis_id,seq_region_id,seq_region_start,seq_region_end,seq_region_strand,display_xref_id,source,description,is_current,canonical_transcript_id,stable_id,version,created_date,modified_date


In [78]:
event_df[event_df['old_stable_id'] == 'ENSG00000281557']

,old_stable_id,old_version,new_stable_id,new_version,mapping_session_id,type,score
2223770,ENSG00000281557,1.0,ENSG00000281557,2.0,407,gene,0.998413
2314091,ENSG00000281557,2.0,ENSG00000281557,3.0,408,gene,0.785448
2354065,ENSG00000281557,3.0,ENSG00000281557,3.0,409,gene,0.900000
2487573,ENSG00000281557,3.0,ENSG00000281557,3.0,410,gene,0.900000
